In [1]:
pip install langchain langchain-core langgraph

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install langchain-openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

os.environ["COHERE_API_KEY"] = "YOUR API KEY"

model = init_chat_model(
    model='command-r7b-12-2024',
    model_provider='cohere'
)

In [4]:
pip install faiss-cpu sentence-transformers langchain-community pypdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

loader = TextLoader("sample_docs.txt")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

C:\Users\priyu\AppData\Local\Temp\ipykernel_4992\2166590445.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
C:\Users\priyu\AppData\Local\Temp\ipykernel_4992\2166590445.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
sample_text = """
LangGraph is a library for building stateful, multi-actor applications with LLMs.
It extends LangChain by allowing you to create graphs where nodes represent steps
and edges represent transitions between them.

RAG stands for Retrieval Augmented Generation. It combines a retriever (which searches
documents) with a generator (an LLM) to produce more accurate, grounded answers.

FAISS is a library by Facebook for efficient similarity search and clustering of dense vectors.
It is commonly used as a vector store in RAG pipelines.
"""

with open("sample_docs.txt", "w") as f:
    f.write(sample_text)

In [7]:
from langchain_core.messages import trim_messages

trimmer = trim_messages(
    max_tokens=1000,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Answer using this context: {context}"),
    MessagesPlaceholder("messages"),
])

In [8]:
def call_model(state: MessagesState):
    last_user_message = state["messages"][-1].content

    retrieved_docs = retriever.invoke(last_user_message)
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    trimmed_messages = trimmer.invoke(state["messages"])

    prompt = prompt_template.invoke({
        "messages": trimmed_messages,
        "context": context
    })

    response = model.invoke(prompt)
    return {"messages": response}

In [18]:
config = {"configurable": {"thread_id": "priyu_test"}}

output1 = chatbot.invoke({"messages": [HumanMessage(content="My name is nick Priyu and my actual name is priyanshu tiwari")]}, config=config)
output1["messages"][-1].pretty_print()

output2 = chatbot.invoke({"messages": [HumanMessage(content="What is my name?")]}, config=config)
output2["messages"][-1].pretty_print()

================================== Ai Message ==================================

Hello Priyanshu! It's great to meet you. I understand that you prefer to go by the name "Nick Priyu." I'll use that name for our interactions unless you tell me otherwise. How can I help you today?
================================== Ai Message ==================================

Your name is Priyanshu Tiwari. You mentioned that you prefer to go by the name "Nick Priyu," but your full name is Priyanshu Tiwari.


In [19]:
workflow = StateGraph(state_schema=MessagesState)
workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

memory = MemorySaver()
chatbot = workflow.compile(checkpointer=memory)

In [20]:
config = {"configurable": {"thread_id": "priyanshu"}}
query = "What is RAG?"
output = chatbot.invoke({"messages": [HumanMessage(content=query)]}, config=config)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

RAG, or Retrieval Augmented Generation, is a technique used in natural language processing and machine learning. It combines two key components: a retriever and a generator.

The retriever is responsible for searching through a large collection of documents or data to find relevant information that might be useful for answering a specific question or completing a task. This is often done using techniques like semantic search or vector search, where the retriever identifies the most similar or relevant pieces of information based on the input query.

The generator, on the other hand, is a language model, typically a large language model (LLM) like those based on transformer architectures. It takes the retrieved information and uses it to generate a response or complete the task. The generator can provide context, facts, or even creative content based on the retrieved data.

RAG is particularly useful in ap

In [21]:
config = {"configurable": {"thread_id": "priyanshu"}}
query = "What is my nick name?"
output = chatbot.invoke({"messages": [HumanMessage(content=query)]}, config=config)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

I don't have personal information about you, so I can't determine your nickname. Nicknames are often given by friends, family, or acquaintances and can vary based on personal relationships, shared experiences, or inside jokes. If you provide more context or details about how you'd like to be referred to, I can offer some suggestions or ideas for a nickname based on that information.


In [22]:
config = {"configurable": {"thread_id": "priyanshu"}}
query = "What is my name?"
output = chatbot.invoke({"messages": [HumanMessage(content=query)]}, config=config)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

I don't have access to personal information about you, so I can't determine your name. If you provide some context or additional details, I can offer some suggestions or ideas based on that information. For example, if you're asking for a nickname, I can provide some ideas based on your interests, personality, or other characteristics.
